In [ ]:
!pip install transformers -q

import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("Done!")

Done!


In [ ]:
from google.colab import files
uploaded = files.upload()  # button aayega — CSV upload karo

df = pd.read_csv('scam_posts_v2.csv')

# Basic check
print(f"Total rows: {len(df)}")
print(f"Scam (1): {df['is_scam'].sum()}")
print(f"Legit (0): {len(df) - df['is_scam'].sum()}")
print(f"Null values: {df['text'].isnull().sum()}")

Saving scam_posts_v2.csv to scam_posts_v2.csv
Total rows: 1150
Scam (1): 650
Legit (0): 500
Null values: 0


In [ ]:
# Tokenizer load karo
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Dataset class — PyTorch ko data dene ka tarika
class JobPostDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            str(self.texts[idx]),
            max_length=128,       # 256 se kam kiya — overfitting rokne ke liye
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Data split karo — 70% train, 15% val, 15% test
X = df['text'].tolist()
y = df['is_scam'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,      # 30% alag karo
    random_state=42,
    stratify=y           # scam/legit ratio same rakho dono mein
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# DataLoader — model ko batch mein data deta hai
train_loader = DataLoader(
    JobPostDataset(X_train, y_train),
    batch_size=32,       # ek baar mein 32 posts — overfitting kam hoga
    shuffle=True
)
val_loader = DataLoader(JobPostDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(JobPostDataset(X_test, y_test), batch_size=32)

print("Data ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train: 805 | Val: 172 | Test: 173
Data ready!


In [ ]:
# GPU use karo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Model load karo
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)

# Optimizer + Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 3
total_steps = len(train_loader) * EPOCHS

# Scheduler — learning rate slowly kam karta hai — overfitting rokta hai
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# Training loop
best_val_acc = 0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        # Gradient clipping — weights bahut bade na ho jaaye
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    preds, actuals = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=1)
            preds.extend(predictions.cpu().numpy())
            actuals.extend(labels.cpu().numpy())

    val_acc = accuracy_score(actuals, preds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.4f}")

    # Best model save karo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  ✅ Best model saved! Accuracy: {val_acc:.4f}")

# --- Final Test ---
model.load_state_dict(torch.load('best_model.pt'))
model.eval()
test_preds, test_actuals = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(predictions.cpu().numpy())
        test_actuals.extend(labels.cpu().numpy())

print("\n=== FINAL TEST RESULTS ===")
print(classification_report(
    test_actuals, test_preds,
    target_names=['Legit', 'Scam']
))

Using: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Loss: 0.4528 | Val Accuracy: 1.0000
  ✅ Best model saved! Accuracy: 1.0000
Epoch 2/3 | Loss: 0.0733 | Val Accuracy: 1.0000
Epoch 3/3 | Loss: 0.0242 | Val Accuracy: 1.0000

=== FINAL TEST RESULTS ===
              precision    recall  f1-score   support

       Legit       0.99      1.00      0.99        75
        Scam       1.00      0.99      0.99        98

    accuracy                           0.99       173
   macro avg       0.99      0.99      0.99       173
weighted avg       0.99      0.99      0.99       173



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_path = '/content/drive/MyDrive/ScamShield/scamshield_model'
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("✅ Model saved to Google Drive!")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Google Drive!
